In [1]:
import torch
from torch.utils.data import Dataset
from torchvision import datasets
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt
import math
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

/home/ailab/.local/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
!nvidia-smi

Failed to initialize NVML: Driver/library version mismatch


In [3]:
torch.__version__

'1.12.1+cu116'

In [4]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

/home/ailab/anaconda3/envs/yolov5/lib/python3.8/site-packages/torch/cuda/__init__.py:83: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 804: forward compatibility was attempted on non supported HW (Triggered internally at  ../c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


In [7]:
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

In [8]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(training_data, batch_size=256, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=256, shuffle=True)

In [9]:
torch.manual_seed(1234)

In [10]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        sm = F.log_softmax(logits, dim =1)
        return sm

model = NeuralNetwork()


In [11]:
model.to(device)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)

In [12]:
learning_rate = 1e-3
batch_size = 256
epochs = 500

In [13]:
# Initialize the loss function
loss_fn = nn.CrossEntropyLoss()

In [14]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [15]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adagrad(model.parameters(), lr=learning_rate)

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.309407  [    0/60000]
loss: 0.955487  [25600/60000]
loss: 0.783849  [51200/60000]
Test Error: 
 Accuracy: 72.5%, Avg loss: 0.792783 

Epoch 2
-------------------------------
loss: 0.740896  [    0/60000]
loss: 0.689182  [25600/60000]
loss: 0.707628  [51200/60000]
Test Error: 
 Accuracy: 76.0%, Avg loss: 0.681956 

Epoch 3
-------------------------------
loss: 0.647786  [    0/60000]
loss: 0.617894  [25600/60000]
loss: 0.555834  [51200/60000]
Test Error: 
 Accuracy: 77.9%, Avg loss: 0.626998 

Epoch 4
-------------------------------
loss: 0.651097  [    0/60000]
loss: 0.590333  [25600/60000]
loss: 0.567029  [51200/60000]
Test Error: 
 Accuracy: 79.0%, Avg loss: 0.601838 

Epoch 5
-------------------------------
loss: 0.556433  [    0/60000]
loss: 0.466785  [25600/60000]
loss: 0.558121  [51200/60000]
Test Error: 
 Accuracy: 79.9%, Avg loss: 0.582630 

Epoch 6
-------------------------------
loss: 0.518977  [    0/60000]
loss: 0.542395  [256